<a href="https://colab.research.google.com/github/ArmFriiz/Dicoding-Submission-FDL/blob/main/Klasifikasi%20Gambar/Template_Submission_Akhir_Klasifikasi_Gambar_Dicoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Proyek Klasifikasi Gambar: [Animals-10: https://www.kaggle.com/datasets/alessiocorrado99/animals10?select=raw-img]**
- **Nama:** Muhammad Faris Akbar
- **Email:** mfarfaris@gmail.com
- **ID Dicoding:** muhammad_faris_akbar

## **Import Semua Packages/Library yang Digunakan**

In [1]:
!pip install split-folders

In [2]:
import kagglehub
import os
import splitfolders
import shutil
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import cv2

from google.colab import files
from PIL import Image
from sklearn.model_selection import train_test_split
from glob import glob
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Conv2D, MaxPooling2D, Flatten, Rescaling, BatchNormalization, RandomFlip, RandomRotation, RandomZoom, Input
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
print(tf.__version__)

2.19.0


## **Data Preparation**

### **Data Loading**

In [ ]:
if not os.path.exists('/root/.kaggle/kaggle.json'):
  files.upload()

In [ ]:
if not os.path.exists('/content/raw-img'):
  !mkdir -p ~/.kaggle/
  !mv kaggle.json ~/.kaggle/
  !chmod 600 ~/.kaggle/kaggle.json

  !kaggle datasets download -d alessiocorrado99/animals10
  !unzip /content/animals10.zip

In [ ]:
fix_path = "/content/raw-img"

In [ ]:
# Dictionary mapping untuk mengganti nama folder
italian_to_english = {
    'cavallo': 'horse',
    'pecora': 'sheep',
    'elefante': 'elephant',
    'gatto': 'cat',
    'scoiattolo': 'squirrel',
    'gallina': 'chicken',
    'ragno': 'spider',
    'mucca': 'cow',
    'cane': 'dog',
    'farfalla': 'butterfly'
}

if italian_to_english["cavallo"] not in os.listdir(fix_path):
  raw_img_path = fix_path

  for italian_name, english_name in italian_to_english.items():
      old_folder_path = os.path.join(raw_img_path, italian_name)
      new_folder_path = os.path.join(raw_img_path, english_name)

      if os.path.exists(old_folder_path):
        os.rename(old_folder_path, new_folder_path)
        print(f"Folder '{italian_name}' berhasil diganti namanya menjadi '{english_name}'")
      else:
        print(f"Folder '{italian_name}' tidak ditemukan di '{raw_img_path}'")

In [ ]:
def print_images_resolution(directory, jumlah_image, total_images, cek_size):
  unique_sizes = set()

  for subdir in os.listdir(directory):
    subdir_path = os.path.join(directory, subdir)
    image_files = os.listdir(subdir_path)
    num_images = len(image_files)
    jumlah_data[subdir] = num_images

    print(f"{subdir}: {num_images}")
    total_images += num_images

    if cek_size:
      for img_file in image_files:
        img_path = os.path.join(subdir_path, img_file)
        with Image.open(img_path) as img:
          unique_sizes.add(img.size)

      for size in unique_sizes:
        print(f"- {size}")
      print("--------------------------------------------------------------------")

  return jumlah_data, total_images

In [ ]:
jumlah_data = {}
total_images = 0

jumlah_data, total_images = print_images_resolution(fix_path, jumlah_data, total_images, cek_size=1)

In [ ]:
print(f"Total ukuran data : {total_images} \n")

plt.figure(figsize=(10, 6))
plt.bar(jumlah_data.keys(), jumlah_data.values())
plt.xlabel('Class')
plt.ylabel('Jumlah Data')
plt.title('Jumlah Data Setiap Kelas')
plt.bar_label(plt.gca().containers[0])
plt.show()

In [ ]:
animal_image = {}

path = fix_path

for i in os.listdir(path):
  animal_image[i] = os.listdir(os.path.join(path, i))

# (len(lung_image.keys()), 5) atau (10, 5)
fig, axs = plt.subplots(len(animal_image.keys()), 5, figsize=(15, 15))

for i, class_name in enumerate(animal_image.keys()):
  images = np.random.choice(animal_image[class_name], 5, replace=False)

  for j, image_name in enumerate(images):
    img_path = os.path.join(path, class_name, image_name)
    img = Image.open(img_path)
    axs[i, j].imshow(img, cmap='gray')
    axs[i, j].set(xlabel=class_name, xticks=[], yticks=[])

fig.tight_layout()

### **Data Preprocessing**

#### **Split Dataset**

Untuk menghindari terjadinya data leakage, splitting dilakukan terlebih dahulu sebelum data augmentasi. Data yang akan diaugmentasi hanya data training sedangkan untuk data testing dan validasi hanya akan dilakukan normalisasi.

In [ ]:
splitfolders.ratio(fix_path, output='dataset_split',
                   seed=1337, ratio=(.8, .1, .1),
                   group_prefix=None, group=None,
                   formats=None, move=False, shuffle=True)

In [ ]:
train_size = {}
val_size = {}
test_size = {}

base_dir = 'dataset_split'
splits = {'train': train_size, 'val': val_size, 'test': test_size}

for split_name, size_dict in splits.items():
    split_path = os.path.join(base_dir, split_name)
    if os.path.exists(split_path):
        for class_name in os.listdir(split_path):
            class_path = os.path.join(split_path, class_name)
            if os.path.isdir(class_path):
                size_dict[class_name] = len(os.listdir(class_path))

In [ ]:
print(f"Total ukuran data training: {sum(train_size.values())} \n")

plt.figure(figsize=(10, 6))
plt.bar(train_size.keys(), train_size.values())
plt.xlabel('Class')
plt.ylabel('Jumlah Data')
plt.title('Jumlah Training Data Setiap Kelas')
plt.bar_label(plt.gca().containers[0])
plt.show()

In [ ]:
print(f"Total ukuran data validation: {sum(val_size.values())} \n")

plt.figure(figsize=(10, 6))
plt.bar(val_size.keys(), val_size.values())
plt.xlabel('Class')
plt.ylabel('Jumlah Data')
plt.title('Jumlah Validation Data Setiap Kelas')
plt.bar_label(plt.gca().containers[0])
plt.show()

In [ ]:
print(f"Total ukuran data testing: {sum(test_size.values())} \n")

plt.figure(figsize=(10, 6))
plt.bar(test_size.keys(), test_size.values())
plt.xlabel('Class')
plt.ylabel('Jumlah Data')
plt.title('Jumlah Testing Data Setiap Kelas')
plt.bar_label(plt.gca().containers[0])
plt.show()

In [ ]:
def balance_dataset(dataset_path, target_count):
  classes = os.listdir(dataset_path)

  datagen = ImageDataGenerator(
      rotation_range=20,
      width_shift_range=0.1,
      height_shift_range=0.1,
      shear_range=0.1,
      zoom_range=0.1,
      horizontal_flip=True,
      fill_mode='nearest'
    )

  for cls in classes:
    class_dir = os.path.join(dataset_path, cls)
    images = glob(os.path.join(class_dir, "*"))
    count = len(images)

    print(f"\nKelas '{cls}': {count} gambar. ")

    if count < target_count:
      diff = target_count - count
      print(f"Menambah {diff} gambar")

      # Generate gambar baru
      aug_generator = datagen.flow_from_directory(
          dataset_path,
          classes=[cls], # Hanya augmentasi untuk kelas yang dipilih
          target_size=(224, 224),
          batch_size=1, # 1 gambar 1 waktu
          save_to_dir=class_dir, # Simpan balik ke foldernya
          save_prefix='aug_bal',
          save_format='jpeg',
          shuffle=False
          )

      # Loop manual untuk stop saat target tercapai
      steps = 0
      for _ in aug_generator:
        steps += 1
        if steps >= diff:
          break

      print(f"Augmentasi data selesai. Total data sebesar {count + steps} data")

In [ ]:
balance_dataset('dataset_split/train', target_count=max(train_size.values()))

In [ ]:
train_size_augmented = {}

base_dir = 'dataset_split'

split_path = os.path.join(base_dir, 'train')
if os.path.exists(split_path):
  for class_name in os.listdir(split_path):
    class_path = os.path.join(split_path, class_name)
    if os.path.isdir(class_path):
      train_size_augmented[class_name] = len(os.listdir(class_path))

In [ ]:
print(f"Total ukuran data training setelah augmentasi: {sum(train_size_augmented.values())} \n")

plt.figure(figsize=(10, 6))
plt.bar(train_size_augmented.keys(), train_size_augmented.values())
plt.xlabel('Class')
plt.ylabel('Jumlah Data')
plt.title('Jumlah Training Data Setiap Kelas Setelah Augmentasi')
plt.bar_label(plt.gca().containers[0])
plt.show()

In [ ]:
dataset_path = 'dataset_split/train'
classes = os.listdir('dataset_split/train')
animal_augmented_image = {}

for cls in classes:
  class_dir = os.path.join(dataset_path, cls)
  augmented_images = glob(os.path.join(class_dir, 'aug_bal_*.jpeg'))

  if len(augmented_images) > 0:
    animal_augmented_image[cls] = augmented_images

# (len(animal_augmented_image.keys()), 5) atau (9, 5)
num_augmented_classes = len(animal_augmented_image.keys())
fig, axs = plt.subplots(num_augmented_classes, 5, figsize=(15, 15))

# Memastikan jika ada data augmentasi minimal 1, lebarkan dimensi
if num_augmented_classes == 1:
    axs = np.expand_dims(axs, axis=0)

for i, class_name in enumerate(animal_augmented_image.keys()):
  images = np.random.choice(animal_augmented_image[class_name], 5, replace=False)
  for j, image_path in enumerate(images):
    img = Image.open(image_path)
    axs[i, j].imshow(img, cmap='gray')
    axs[i, j].set(xlabel=class_name, xticks=[], yticks=[])

fig.tight_layout()

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Load Train
train_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset_split/train',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True
)

# Load Val
val_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset_split/val',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Load Test
test_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset_split/test',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

In [ ]:
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Kelas: {class_names}")
print(f"Jumlah kelas: {num_classes}")

In [ ]:
# Optimasi Performa
AUTOTUNE = tf.data.AUTOTUNE

if not os.path.exists('cache'):
    os.makedirs('cache')

train_ds = train_ds.cache('cache/train_cache').shuffle(1000).prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.cache('cache/val_cache').prefetch(buffer_size=AUTOTUNE)

test_ds = test_ds.cache('cache/test_cache').prefetch(buffer_size=AUTOTUNE)

## **Modelling**

In [ ]:
class myCallback(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs={}):
    accuracy = logs.get('accuracy')
    val_accuracy = logs.get('val_accuracy')

    if accuracy is not None and accuracy > 0.95 and val_accuracy > 0.95:
      print("\nAccuracy dan validation_accuracy > 95%. Menghentikan training.")
      self.model.stop_training = True

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', # Memantau validasi loss
    factor=0.2,         # Mengurangi learning rate menjadi 20% dari semula
    patience=3,         # Jika val_loss tidak membaik setelah 3 epoch
    min_lr=0.00001      # Learning rate minimum
)

callbacks = [myCallback(), reduce_lr]

In [ ]:
base_model = MobileNetV3Large(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
    include_preprocessing=True
)

base_model.trainable = False

model = Sequential([
    Input(shape=(224, 224, 3)),
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.1),

    base_model,

    Conv2D(512, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    # Dropout(0.2),
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
history_cnn = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
loss, accuracy = model.evaluate(test_ds)
print(f"Test Accuracy: {accuracy*100:.2f}%")

## **Evaluasi dan Visualisasi**

In [ ]:
plt.plot(history_cnn.history['loss'])
plt.plot(history_cnn.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

In [ ]:
plt.plot(history_cnn.history['accuracy'])
plt.plot(history_cnn.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

## **Konversi Model**

In [ ]:
# os.makedirs('model', exist_ok=True)

In [ ]:
# # tf.saved_model.save(model, 'model/saved_model')
# model.export('model/saved_model')

In [ ]:
# os.makedirs('model/tflite', exist_ok=True)

In [ ]:
# export_dir = 'model/tflite'
# # tf.saved_model.save(model, export_dir)

# converter = tf.lite.TFLiteConverter.from_saved_model('/content/model/saved_model')
# converter.target_spec.supported_ops = [
#     tf.lite.OpsSet.TFLITE_BUILTINS, # Operasi standar TFLite
#     tf.lite.OpsSet.SELECT_TF_OPS   # Mendukung operasi TF seperti StatelessRandom
# ]
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# # Jalankan konversi
# tflite_model = converter.convert()

# tflite_model_file = pathlib.Path('model/tflite/model.tflite')
# tflite_model_file.write_bytes(tflite_model)

In [ ]:
# model.save('model.keras')

# !tensorflowjs_converter \
#     --input_format=keras_keras \
#     --output_format=tfjs_layers_model \
#     model.keras \
#     /content/model/tfjs_model

## **Inference (Optional)**